# 🛰️ Aula 07 — Conceito e Projeto de Soft-Sensors

**Disciplina:** Inteligência Artificial Aplicada à Engenharia Química  
**Dataset:** `coluna_destilacao_30dias.csv` — 30 dias de operação (8 variáveis)

---

## Contexto

Você é o engenheiro de processos e recebeu a missão: **projetar um soft-sensor para estimar a composição do destilado em tempo real**, permitindo reduzir a frequência do GC (cromatógrafo) de 30 min para 2h.

## 3.1 — Projeto de Soft-Sensor: 5 Etapas

Siga as 5 etapas abaixo, cada uma em uma seção do notebook.

### Etapa 1: Definir o problema

In [ ]:
print("=== ETAPA 1: DEFINIÇÃO ===")
print("Target (y): composicao_destilado (% mol etanol)")
print("RMSE alvo:   < 1.0% (definido pelo gerente)")
print("Frequência:  predição a cada 1 min")
print("GC atual:    análise a cada 30 min → quer reduzir para 2h")

### Etapa 2: Selecionar variáveis secundárias (features)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

print("=== ETAPA 2: SELEÇÃO DE FEATURES ===")
feature_cols = [
    'T_top_C', 'T_base_C', 'P_coluna_kPa',
    'R_refluxo', 'F_alimentacao_kg_h'
]
print(f"Features selecionadas: {feature_cols}")
print("Justificativa:")
print("  • T_top_C  → equilíbrio líquido-vapor etanol/água (mais etanol = T mais baixa)")
print("  • R_refluxo → controle direto da pureza do topo")
print("  • P_coluna → desloca o ponto de bolha (equilíbrio)")
print("  • T_base   → condição do fundo da coluna")
print("  • F_alimentação → carga do processo")

### Etapa 3: Preparar dados (lags + alinhar)

In [ ]:
print("=== ETAPA 3: PREPARAÇÃO ===")

URL = "https://raw.githubusercontent.com/LuisGSVasconcelos/IA_EngQuimica/main/dados/aula04/coluna_destilacao_30dias.csv"
df = pd.read_csv(URL, parse_dates=['timestamp'])
df.set_index('timestamp', inplace=True)

# Criar lags (atraso de transporte) — 1 e 3 passos atrás
df['T_top_lag1'] = df['T_top_C'].shift(1)
df['T_top_lag3'] = df['T_top_C'].shift(3)
df['R_lag1'] = df['R_refluxo'].shift(1)
df['R_lag3'] = df['R_refluxo'].shift(3)

df.dropna(inplace=True)

X = df[feature_cols + ['T_top_lag1', 'T_top_lag3', 'R_lag1', 'R_lag3']]
y = df['composicao_destilado']

print(f"Dados preparados: {X.shape[0]} amostras, {X.shape[1]} features")

### Etapa 4: Modelar (RF + XGBoost)

In [ ]:
print("=== ETAPA 4: MODELAGEM ===")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelos = {
    'Random Forest (100)': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost (100)': XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
}

resultados = []
for nome, model in modelos.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_root_mean_squared_error')
    cv_rmse = -cv_scores.mean()
    ok = '✅ SIM' if rmse < 1.0 else '❌ NÃO'
    resultados.append({'Modelo': nome, 'RMSE teste (%)': round(rmse, 3),
                      'RMSE CV (%)': round(cv_rmse, 3), 'Alvo <1%': ok})
    print(f"{nome}:")
    print(f"  RMSE teste: {rmse:.3f}%  |  RMSE CV: {cv_rmse:.3f}%  |  Alvo: {ok}\n")

print(pd.DataFrame(resultados).to_string(index=False))

### Etapa 5: Avaliar viabilidade + implantação (conceitual)

In [ ]:
print("=== ETAPA 5: VIABILIDADE E IMPLANTAÇÃO ===")
print("Conclusão:")
print("  • Se RMSE < 1.0% → soft-sensor é VIÁVEL")
print("  • Recomendação: implantar com predição a cada 1 min")
print("  • Manter GC a cada 2h para calibração e detecção de drift")
print("  • Retreinar modelo mensalmente com novos dados do GC")

### ✏️ Pausa reflexiva (2 min)

O modelo que treinamos hoje vs o da Aula 5 — a diferença é o **propósito** (predição acadêmica vs sensor industrial). O que muda na prática?

> _Escreva aqui..._

---

## 3.2 — Exercício em Grupo: Viabilidade Técnica e Econômica

Cada grupo avalia um caso diferente: **soft-sensor é viável?** (técnica + economicamente)

| Grupo | Processo | Variável | RMSE alvo | Dados | Custo analisador físico |
|-------|----------|----------|-----------|-------|-------------------------|
| **A** | Coluna destilação | Composição topo | ±0.5% | 2 anos PIMS | GC = R$ 200k + R$ 30k/ano |
| **B** | Reator CSTR | Conversão | ±1.0% | 6 meses | Lab = R$ 15k/ano |
| **C** | Trocador calor | Coeficiente U | ±5% | 3 meses | Manual = R$ 0 |
| **D** | Secador spray | Umidade | ±0.2% | 1 ano (falhas) | NIR = R$ 150k + R$ 10k/ano |

In [ ]:
# Matriz de decisão do seu grupo
decisao = {
    'Grupo': '___',
    'Processo': '___',
    'Dados suficientes para treinar?': '___',
    'RMSE alvo factível?': '___',
    'Custo justifica investimento?': '___',
    'Decisão final': '___'
}
for k, v in decisao.items():
    print(f"{k}: {v}")

### 🧠 Desafio extra (NT)

Para o grupo C (coeficiente U do trocador), o cálculo manual é grátis mas impreciso. Quanto você pagaria por um soft-sensor que estima U com erro < 5%?

> _Escreva aqui..._

---

## Checklist de Projeto de Soft-Sensor

- [ ] Variável primária definida
- [ ] Features candidatas listadas e justificadas
- [ ] Dados históricos avaliados (≥6 meses?)
- [ ] RMSE alvo definido
- [ ] 2 modelos treinados e comparados
- [ ] Custo do analisador físico anual calculado
- [ ] Custo do soft-sensor estimado
- [ ] Decisão de implantação documentada